In [0]:
# ── Cell 1: Load silver ───────────────────────────────────────────────────
storage_account = "onlinelearningd1"
key = dbutils.secrets.get(scope="adls-scope", key="storage-key")
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", key)

from pyspark.sql import functions as F

df = spark.read.parquet(
    f"abfss://silver@{storage_account}.dfs.core.windows.net/clean/"
)
print(f"Silver rows loaded: {df.count()}")

Silver rows loaded: 1163


In [0]:
# ── Cell 2: Gold Table 1 — by education level ─────────────────────────────
gold_edu = (df.groupBy("education_level")
    .agg(
        F.count("user_id").alias("total_users"),
        F.round(F.avg("satisfaction_rating"), 2).alias("avg_satisfaction"),
        F.round(F.avg("quiz_score_avg"), 2).alias("avg_quiz_score"),
        F.round(F.avg("hours_spent_weekly"), 2).alias("avg_hours_weekly"),
        F.round(F.avg("login_frequency"), 2).alias("avg_logins"),
        F.round(F.avg("engagement_score"), 2).alias("avg_engagement"),
        F.round(F.sum("completed_binary") / F.count("user_id") * 100, 1)
         .alias("completion_rate_pct")
    )
)
display(gold_edu)

education_level,total_users,avg_satisfaction,avg_quiz_score,avg_hours_weekly,avg_logins,avg_engagement,completion_rate_pct
High School,246,3.01,68.79,10.57,7.86,9.22,50.4
Master,244,2.91,70.13,10.78,7.47,9.13,44.3
PhD,231,2.96,67.93,10.53,7.45,8.99,46.3
Other,244,3.07,69.95,10.63,7.61,9.12,45.5
Bachelor,198,3.06,67.33,11.07,7.49,9.28,47.5


In [0]:
# ── Cell 3: Gold Table 2 — by device ─────────────────────────────────────
gold_device = (df.groupBy("preferred_device")
    .agg(
        F.count("user_id").alias("total_users"),
        F.round(F.avg("quiz_score_avg"), 2).alias("avg_quiz_score"),
        F.round(F.avg("satisfaction_rating"), 2).alias("avg_satisfaction"),
        F.round(F.avg("hours_spent_weekly"), 2).alias("avg_hours_weekly"),
        F.round(F.sum("completed_binary") / F.count("user_id") * 100, 1)
         .alias("completion_rate_pct")
    )
)
display(gold_device)

preferred_device,total_users,avg_quiz_score,avg_satisfaction,avg_hours_weekly,completion_rate_pct
Mobile,378,68.54,3.07,10.71,42.1
Tablet,418,68.98,3.1,10.8,51.9
Desktop,367,69.17,2.81,10.6,45.8


In [0]:
# ── Cell 4: Gold Table 3 — by country ────────────────────────────────────
gold_country = (df.groupBy("country")
    .agg(
        F.count("user_id").alias("total_users"),
        F.round(F.avg("satisfaction_rating"), 2).alias("avg_satisfaction"),
        F.round(F.avg("quiz_score_avg"), 2).alias("avg_quiz_score"),
        F.round(F.sum("completed_binary") / F.count("user_id") * 100, 1)
         .alias("completion_rate_pct")
    )
)
display(gold_country)

country,total_users,avg_satisfaction,avg_quiz_score,completion_rate_pct
India,230,3.0,67.57,42.2
Unknown,35,3.09,69.93,45.7
USA,222,3.11,68.27,45.9
UK,192,3.1,67.9,51.0
Canada,251,2.76,71.01,43.0
Australia,233,3.05,69.18,52.8


In [0]:
# ── Cell 5: Gold Table 4 — satisfaction group vs outcomes ─────────────────
gold_sat = (df.groupBy("satisfaction_group")
    .agg(
        F.count("user_id").alias("total_users"),
        F.round(F.avg("quiz_score_avg"), 2).alias("avg_quiz_score"),
        F.round(F.avg("hours_spent_weekly"), 2).alias("avg_hours_weekly"),
        F.round(F.sum("completed_binary") / F.count("user_id") * 100, 1)
         .alias("completion_rate_pct")
    )
)
display(gold_sat)

satisfaction_group,total_users,avg_quiz_score,avg_hours_weekly,completion_rate_pct
High,441,69.63,10.39,45.6
Low,453,68.95,10.91,47.9
Medium,269,67.61,10.87,46.8


In [0]:
# ── Cell 6: Gold Table 5 — study band vs outcomes ─────────────────────────
gold_study = (df.groupBy("study_band")
    .agg(
        F.count("user_id").alias("total_users"),
        F.round(F.avg("quiz_score_avg"), 2).alias("avg_quiz_score"),
        F.round(F.avg("satisfaction_rating"), 2).alias("avg_satisfaction"),
        F.round(F.sum("completed_binary") / F.count("user_id") * 100, 1)
         .alias("completion_rate_pct")
    )
    .orderBy("study_band")
)
display(gold_study)

study_band,total_users,avg_quiz_score,avg_satisfaction,completion_rate_pct
0-5 hrs,201,68.59,3.08,42.3
10-15 hrs,377,70.03,3.03,46.4
15+ hrs,293,69.34,2.8,49.5
5-10 hrs,292,67.19,3.1,47.6


In [0]:
# ── Cell 7: Write all gold tables ─────────────────────────────────────────
base = f"abfss://gold@{storage_account}.dfs.core.windows.net"
gold_edu.write.mode("overwrite").parquet(f"{base}/by_education/")
gold_device.write.mode("overwrite").parquet(f"{base}/by_device/")
gold_country.write.mode("overwrite").parquet(f"{base}/by_country/")
gold_sat.write.mode("overwrite").parquet(f"{base}/by_satisfaction/")
gold_study.write.mode("overwrite").parquet(f"{base}/by_study_band/")
print("All gold tables written.")

All gold tables written.
